In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

DATASET_PATH = "pos_tagged_dataset.csv"

df = pd.read_csv(DATASET_PATH)

SENTENCE_COLUMN = "sentence_id"
WORD_COLUMN = "word"
TAG_COLUMN = "tag"

df = df.dropna(
    subset=[
        SENTENCE_COLUMN,
        WORD_COLUMN,
        TAG_COLUMN
    ]
)

df[WORD_COLUMN] = df[WORD_COLUMN].astype(str).str.lower()
df[TAG_COLUMN] = df[TAG_COLUMN].astype(str)

sentences = []

for sentence_id, group in df.groupby(SENTENCE_COLUMN):

    words = group[WORD_COLUMN].tolist()
    tags = group[TAG_COLUMN].tolist()

    if len(words) > 0:
        sentences.append((words, tags))

train_data, test_data = train_test_split(
    sentences,
    test_size=0.2,
    random_state=42
)

tags = sorted(
    list(
        set(
            tag
            for sentence, tag_sequence in train_data
            for tag in tag_sequence
        )
    )
)

tag_to_index = {
    tag: index
    for index, tag in enumerate(tags)
}

index_to_tag = {
    index: tag
    for tag, index in tag_to_index.items()
}

vocabulary = set(
    word
    for sentence, tag_sequence in train_data
    for word in sentence
)

num_tags = len(tags)

initial_counts = np.zeros(num_tags)

transition_counts = np.zeros(
    (num_tags, num_tags)
)

emission_counts = {}

for word in vocabulary:
    emission_counts[word] = np.zeros(num_tags)

for words, tag_sequence in train_data:

    first_tag = tag_sequence[0]

    first_tag_index = tag_to_index[first_tag]

    initial_counts[first_tag_index] += 1

for words, tag_sequence in train_data:

    for i in range(len(tag_sequence) - 1):

        current_tag = tag_sequence[i]
        next_tag = tag_sequence[i + 1]

        current_index = tag_to_index[current_tag]
        next_index = tag_to_index[next_tag]

        transition_counts[
            current_index,
            next_index
        ] += 1

for words, tag_sequence in train_data:

    for word, tag in zip(words, tag_sequence):

        tag_index = tag_to_index[tag]

        emission_counts[word][tag_index] += 1

alpha = 1.0

initial_probabilities = (
    initial_counts + alpha
) / (
    np.sum(initial_counts)
    + alpha * num_tags
)

transition_probabilities = (
    transition_counts + alpha
) / (
    transition_counts.sum(
        axis=1,
        keepdims=True
    )
    + alpha * num_tags
)

emission_probabilities = {}

for word, counts in emission_counts.items():

    emission_probabilities[word] = (
        counts + alpha
    ) / (
        counts.sum()
        + alpha * num_tags
    )

log_initial_probabilities = np.log(
    initial_probabilities
)

log_transition_probabilities = np.log(
    transition_probabilities
)

log_emission_probabilities = {
    word: np.log(probabilities)
    for word, probabilities
    in emission_probabilities.items()
}

unknown_word_probability = np.log(
    alpha / (alpha * num_tags)
)

def viterbi(words):

    sequence_length = len(words)

    if sequence_length == 0:
        return []

    first_word = words[0]

    if first_word in log_emission_probabilities:

        first_emission = (
            log_emission_probabilities[first_word]
        )

    else:

        first_emission = np.full(
            num_tags,
            unknown_word_probability
        )

    viterbi_matrix = np.zeros(
        (sequence_length, num_tags)
    )

    backpointer_matrix = np.zeros(
        (sequence_length, num_tags),
        dtype=int
    )

    viterbi_matrix[0] = (
        log_initial_probabilities
        + first_emission
    )

    for t in range(1, sequence_length):

        word = words[t]

        if word in log_emission_probabilities:

            emission = (
                log_emission_probabilities[word]
            )

        else:

            emission = np.full(
                num_tags,
                unknown_word_probability
            )

        scores = (
            viterbi_matrix[t - 1][:, None]
            + log_transition_probabilities
        )

        backpointer_matrix[t] = np.argmax(
            scores,
            axis=0
        )

        viterbi_matrix[t] = (
            np.max(scores, axis=0)
            + emission
        )

    best_last_tag = np.argmax(
        viterbi_matrix[-1]
    )

    best_path = [
        best_last_tag
    ]

    for t in range(
        sequence_length - 1,
        0,
        -1
    ):

        best_last_tag = backpointer_matrix[
            t,
            best_last_tag
        ]

        best_path.append(
            best_last_tag
        )

    best_path.reverse()

    return [
        index_to_tag[index]
        for index in best_path
    ]

actual_tags = []
predicted_tags = []

for words, true_tags in test_data:

    predicted = viterbi(words)

    actual_tags.extend(true_tags)
    predicted_tags.extend(predicted)

accuracy = accuracy_score(
    actual_tags,
    predicted_tags
)

precision = precision_score(
    actual_tags,
    predicted_tags,
    average="weighted",
    zero_division=0
)

recall = recall_score(
    actual_tags,
    predicted_tags,
    average="weighted",
    zero_division=0
)

f1 = f1_score(
    actual_tags,
    predicted_tags,
    average="weighted",
    zero_division=0
)

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1-Score:", f1)

print(
    classification_report(
        actual_tags,
        predicted_tags,
        zero_division=0
    )
)

unseen_sentences = [
    "the cat sat on the mat",
    "students are learning artificial intelligence",
    "the computer processes data quickly",
    "she reads a book every day",
    "machine learning models predict outcomes"
]

for sentence in unseen_sentences:

    words = sentence.lower().split()

    predicted = viterbi(words)

    print("\nSentence:", sentence)

    for word, tag in zip(words, predicted):

        print(
            word,
            "->",
            tag,
            end=" | "
        )

    print()